In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

root = Path.home() / "projects" / "loan-default-risk"
df = pd.read_parquet(root / "data" / "processed" / "loans_resolved.parquet")
print(df.shape)

(1345310, 153)


In [2]:
df["issue_dt"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["issue_year"] = df["issue_dt"].dt.year
df["split"] = np.select(
    [df["issue_year"] <= 2015, df["issue_year"] == 2016, df["issue_year"] == 2017],
    ["train", "valid", "test"], default="unused")
print(df.groupby("split")["target"].agg(["count", "mean"]).round(4))
df = df[df["split"] != "unused"].copy()

         count    mean
split                 
test    169300  0.2312
train   826604  0.1843
unused   56311  0.1575
valid   293095  0.2328


In [3]:
df["term_months"] = df["term"].str.extract(r"(\d+)")[0].astype(float)
df["emp_length_yrs"] = df["emp_length"].str.extract(r"(\d+)")[0].astype(float)
df.loc[df["emp_length"] == "< 1 year", "emp_length_yrs"] = 0
df["credit_history_yrs"] = df["issue_year"] - pd.to_datetime(df["earliest_cr_line"], format="%b-%Y").dt.year
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
df["loan_to_income"] = df["loan_amnt"] / df["annual_inc"].replace(0, np.nan)
df["revol_bal_to_income"] = df["revol_bal"] / df["annual_inc"].replace(0, np.nan)
df["open_to_total_acc"] = df["open_acc"] / df["total_acc"].replace(0, np.nan)
df[["term_months", "emp_length_yrs", "credit_history_yrs", "fico_avg", "loan_to_income"]].describe().round(2)

,term_months,emp_length_yrs,credit_history_yrs,fico_avg,loan_to_income
count,1288999.00,1215422.00,1288999.00,1288999.00,1288786.00
mean,41.76,5.97,16.29,697.65,0.36
std,10.25,3.69,7.50,31.45,62.58
min,36.00,0.00,3.00,627.00,0.00
25%,36.00,2.00,11.00,672.00,0.12
50%,36.00,6.00,15.00,692.00,0.20
75%,36.00,10.00,20.00,712.00,0.29
max,60.00,10.00,83.00,847.50,40000.00


In [4]:
leak = [
    "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee", "last_pymnt_d", "last_pymnt_amnt",
    "next_pymnt_d", "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low",
    "debt_settlement_flag", "debt_settlement_flag_date", "settlement_status", "settlement_date",
    "settlement_amount", "settlement_percentage", "settlement_term",
    "hardship_flag", "hardship_type", "hardship_reason", "hardship_status", "deferral_term",
    "hardship_amount", "hardship_start_date", "hardship_end_date", "payment_plan_start_date",
    "hardship_length", "hardship_dpd", "hardship_loan_status",
    "orig_projected_additional_accrued_interest", "hardship_payoff_balance_amount",
    "hardship_last_payment_amount", "funded_amnt", "funded_amnt_inv", "pymnt_plan",
    "policy_code", "loan_status",
]
lender_score = ["int_rate", "grade", "sub_grade", "installment"]
ids_text = ["id", "member_id", "url", "desc", "title", "emp_title", "zip_code"]
raw_used = ["term", "emp_length", "earliest_cr_line", "fico_range_low", "fico_range_high", "issue_d", "issue_dt"]

drop_cols = leak + lender_score + ids_text + raw_used
print("not found:", [c for c in drop_cols if c not in df.columns])
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(df.shape)

not found: []
(1288999, 102)


In [5]:
miss_train = df.loc[df["split"] == "train"].isna().mean()
too_missing = miss_train[miss_train > 0.5].index.tolist()
print(len(too_missing), "columns dropped for >50% missing in train")
df = df.drop(columns=too_missing)
print(df.shape)

35 columns dropped for >50% missing in train
(1288999, 67)


In [6]:
from sklearn.metrics import roc_auc_score

tr = df[df["split"] == "train"]
nums = tr.select_dtypes("number").columns.drop(["target", "issue_year"])
aucs = {}
for c in nums:
    s = tr[[c, "target"]].dropna()
    if s[c].nunique() > 1:
        a = roc_auc_score(s["target"], s[c])
        aucs[c] = max(a, 1 - a)
pd.Series(aucs).sort_values(ascending=False).head(10).round(4)

term_months             0.6128
loan_to_income          0.6015
fico_avg                0.5914
acc_open_past_24mths    0.5822
dti                     0.5784
num_tl_op_past_12m      0.5701
bc_open_to_buy          0.5640
mo_sin_rcnt_tl          0.5565
total_bc_limit          0.5530
avg_cur_bal             0.5512
dtype: float64

In [7]:
print(df.dtypes.value_counts())
print(df.select_dtypes("object").nunique())
df.to_parquet(root / "data" / "processed" / "features.parquet", index=False)
print("saved", df.shape)

float64    56
str         8
int32       2
int64       1
Name: count, dtype: int64
home_ownership          6
verification_status     3
purpose                14
addr_state             51
initial_list_status     2
application_type        2
disbursement_method     2
split                   3
dtype: int64


C:\Users\gdevi\AppData\Local\Temp\ipykernel_7644\3131966135.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes("object").nunique())


saved (1288999, 67)


In [8]:
for c in ["loan_to_income", "revol_bal_to_income"]:
    cap = df.loc[df["split"] == "train", c].quantile(0.999)
    df[c] = df[c].clip(upper=cap)
    print(c, "capped at", round(cap, 3), "| new max", round(df[c].max(), 3))

df.to_parquet(root / "data" / "processed" / "features.parquet", index=False)
print("saved", df.shape)

loan_to_income capped at 0.5 | new max 0.5
revol_bal_to_income capped at 2.067 | new max 2.067
saved (1288999, 67)
